# Part e: Classification on MNIST


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # add Code/ so `src` can be imported

In [ ]:
import numpy as np
from src.data import load_mnist, one_hot, stratified_kfold_indices
from src.metrics import accuracy, confusion_matrix_multiclass, per_class_from_confusion, standard_error
from src.network import NeuralNetwork
from src.functions import softmax, ReLU, ReLU_der, identity_der, cross_entropy, softmax_cross_entropy_der
from src.optimizers import Adam
from src.plotting import heatmap

# Loading MNIST data and splitting it into training and test data

Subsets are created to be used in training and finding optimal hyperparameters in order to lower computational cost. 

These subsets are made so that each digit is representet equally

In [ ]:
# from week 44 exercises, exercise 3
x_train, y_train, x_test, y_test = load_mnist("../data/mnist.npz") # loading the mnist dataset, flattening the image pixel to shape (784,)

rng = np.random.default_rng(2026) # setting a seed for reproducibility

# for every digit, find 1000 random positions of images that is this digit. Join the 1000 image positions for every digit in one long array
# This makes every digit equally represented
sub = np.concatenate([rng.choice(np.where(y_train == c)[0], 1000, replace=False) for c in range(10)])
X_sub, y_sub = x_train[sub], y_train[sub] # pick out these randomly chosen images by using their positions (sub)

# Creating the network with one hidden layer and finding optimal hyperparameters

Training network using cross validation on a subset of the data

Finding optimal hyperparamers to be used later for the whole dataset, choose based on highest accuracy of network

In [ ]:
from sklearn.metrics import accuracy_score
# from week 44 exercises, exercise 4
def cross_validate_by_hand(fit_predict, X, y, folds, score, scale=True):
    """One validation score and one training score per fold; the scaler is fitted on the training part of each fold only."""
    val_scores, train_scores = [], []
    for train_idx, val_idx in folds:
        X_tr, X_val = X[train_idx], X[val_idx]
        if scale:
            mean, std = X_tr.mean(axis=0), X_tr.std(axis=0)        # fitted on the training part ...
            X_tr, X_val = (X_tr - mean) / std, (X_val - mean) / std   # ... applied to both
        pred_val, pred_tr = fit_predict(X_tr, y[train_idx], X_val, X_tr)
        val_scores.append(score(y[val_idx], pred_val))
        train_scores.append(score(y[train_idx], pred_tr))
    return np.array(val_scores), np.array(train_scores)

In [ ]:
# Defining a function that trains a network and returns its predictions
# structure from week 44 exercise 4, using our own NeuralNetwork class
def fit_predict_net(X_tr, y_tr, X_val, X_tr_again, gamma=1e-3, lmbda=0.0):
    '''
    Functions used:
    ReLU make negative values 0 and positive values 1.
    softmax turns the 10 outputs into probabilities that sum to 1.
    cross_entropy is the cost function for classification.
    softmax_cross_entropy_der is the derivative used in backpropagation for the last layer. 
    identity_der is a derivative that is always 1
    The Adam optimizer updates the weights during training.
    '''
    # Our own NeuralNetwork (784 -> 100 -> 10, ReLU, softmax + cross entropy)
    net = NeuralNetwork(784, [100, 10], [ReLU, softmax], [ReLU_der, identity_der], # creating a new network with our NeuralNetwork class
                        cross_entropy, softmax_cross_entropy_der, Adam(gamma=gamma),
                        regularization="L2", lmbda=lmbda)
    net.train(X_tr, one_hot(y_tr, 10), epochs=5, batch_size=100, rng=np.random.default_rng(2026)) # training the network on one-hot targets

    # Return the predicted digits (argmax of the softmax output)
    return np.argmax(net.predict(X_val), axis=1), np.argmax(net.predict(X_tr_again), axis=1)

In [ ]:
# from week 44 exercise 4, with changes according to our NeuralNetwork
# Claude helped pick gamma and lambda values to investigate

gammas, lambdas = [1e-4, 1e-3, 1e-2, 3e-2, 1e-1], [0.0, 1e-5, 1e-4, 1e-3]          # The hyperparameters that will be investigated 
folds = stratified_kfold_indices(y_sub, 5, np.random.default_rng(2026))            # stratified version, to make sure every fold has the same number of each digit
table = np.zeros((len(gammas), len(lambdas)))
table_se = np.zeros_like(table)
for i, g in enumerate(gammas):
    for j, l in enumerate(lambdas):
        fp = lambda X_tr, y_tr, X_val, X_tr2, g=g, l=l: fit_predict_net(X_tr, y_tr, X_val, X_tr2, g, l)   # Using our fit_predict_net function that predicts digits for network with a hidden layer
        v, _ = cross_validate_by_hand(fp, X_sub, y_sub, folds, accuracy_score, scale=False)           # only looking at a MNIST subset defined earlier, specifically looking at accuracy, no scaling since pixels are already scaled in MNIST (also prevents NaN for black pixels (0))
        table[i, j], table_se[i, j] = v.mean(), v.std(ddof=1) / np.sqrt(len(v))
print("CV accuracy (rows gamma, columns lambda):\n", np.round(table, 4))
print("standard error:\n", np.round(table_se, 4))

# Visualize the results in a heatmap
heatmap(table, lambdas, gammas, r"$\lambda$", r"$\gamma$", label="CV accuracy", name="e_heatmap_l2")

In [ ]:
# choosing gamma and lambda: the best cell, and the one-standard-error rule
# from week 44 Tuesday notebook, changes are commented
shape = (len(gammas), len(lambdas))
i_best = np.unravel_index(np.argmax(table), shape)               # argmax (accuracy) as this optimizes our classification network
within = table >= table[i_best] - table_se[i_best]               #  >= best - s.e., instead of <= best + s.e, due to using argmax and not argmin as for MSE cost functions

# the one-standard-error rule: among the cells within one s.e., the largest penalty, then the smallest learning rate
cand = [(j, i) for i in range(shape[0]) for j in range(shape[1]) if within[i, j]]
j1, i1 = max(cand, key=lambda t: (lambdas[t[0]], -gammas[t[1]]))

print(f"best: gamma {gammas[i_best[0]]:g}, lambda {lambdas[i_best[1]]:g}: {table[i_best]:.4f} +/- {table_se[i_best]:.4f}")
print("within one standard error of the best:\n", within)
print(f"one-standard-error rule: gamma {gammas[i1]:g}, lambda {lambdas[j1]:g}: {table[i1, j1]:.4f}")

# Now looking at accuracy for each hyperparameter combination with logistic regression, no hidden layer

Using the same principle as for neural network with hidden layer, adjusting it to be without a layer

In [ ]:
# based on fit_predict_net, without the hidden layer
def fit_predict_logreg(X_tr, y_tr, X_val, X_tr_again, gamma=1e-3, lmbda=0.0):
    # changed from fit_predict_net: logistic regression, 784 -> 10 (no hidden layer), softmax + cross entropy
    net = NeuralNetwork(784, [10], [softmax], [identity_der],
                        cross_entropy, softmax_cross_entropy_der, Adam(gamma=gamma),
                        regularization="L2", lmbda=lmbda)
    net.train(X_tr, one_hot(y_tr, 10), epochs=5, batch_size=100, rng=np.random.default_rng(2026))
    return np.argmax(net.predict(X_val), axis=1), np.argmax(net.predict(X_tr_again), axis=1)

In [ ]:
# Using the same cross-validation but with fit_predict_logreg now

table_lr = np.zeros((len(gammas), len(lambdas)))
table_lr_se = np.zeros_like(table_lr)
for i, g in enumerate(gammas):
    for j, l in enumerate(lambdas):
        fp = lambda X_tr, y_tr, X_val, X_tr2, g=g, l=l: fit_predict_logreg(X_tr, y_tr, X_val, X_tr2, g, l)   # fit_predict_logreg
        v, _ = cross_validate_by_hand(fp, X_sub, y_sub, folds, accuracy_score, scale=False)
        table_lr[i, j], table_lr_se[i, j] = v.mean(), v.std(ddof=1) / np.sqrt(len(v))
print("logistic regression, CV accuracy (rows gamma, columns lambda):\n", np.round(table_lr, 4))
print("standard error:\n", np.round(table_lr_se, 4))

# Final analysis on whole dataset

Training on the whole dataset (60 000 training images), both for a neural network with a hidden layer and logistic regression

Tested once on 10 000 test images

Visualizing the results of the neural network in a confusion matrix and as per-class precision and recall

In [ ]:

# final models: trained once on all 60 000 training images, tested once on the 10 000 test images
# Claude helped debugging when calling these functions with the whole dataset:
pred_test_net, pred_train_net = fit_predict_net(x_train, y_train, x_test, x_train, gamma=0.01, lmbda=0.0)
pred_test_lr, pred_train_lr = fit_predict_logreg(x_train, y_train, x_test, x_train, gamma=0.01, lmbda=1e-4)

# from week 43 Tuesday notebook
for name, pred in (("logistic regression", pred_test_lr), ("one hidden layer of 100", pred_test_net)): # for each type of classifier and corresponding predictions:
    acc = accuracy_score(y_test, pred) # find the accuracy
    print(f"{name}: test accuracy {acc:.4f} +/- {standard_error(acc, len(y_test)):.4f}") # print the accuracy +/- standard error

In [ ]:

import matplotlib.pyplot as plt
from src.plotting import BLUE, RED, save_fig
from src.metrics import per_class_from_confusion 

# from week 43 Monday notebook. name/save_fig were debugged with Claude
def fig_confusion10(sm, name=None): 
    """The 10 x 10 confusion matrix and the per-class precision and recall (week 43 Monday, fig_confusion10)."""
    C = sm["C"]
    prec, rec = per_class_from_confusion(C)
    fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.6), gridspec_kw={"width_ratios": [1.15, 1]})
    off = C.astype(float).copy()
    np.fill_diagonal(off, np.nan)
    im = ax[0].imshow(off, cmap="Reds", vmin=0, vmax=np.nanmax(off))
    for i in range(10):
        for j in range(10):
            if i == j:
                ax[0].text(j, i, f"{C[i, j]}", ha="center", va="center", fontsize=7, color=BLUE, fontweight="bold")
            elif C[i, j] > 0:
                ax[0].text(j, i, f"{C[i, j]}", ha="center", va="center", fontsize=7,
                           color="white" if C[i, j] > 0.6 * np.nanmax(off) else "black")
    ax[0].set_xticks(range(10))
    ax[0].set_yticks(range(10))
    ax[0].set_xlabel("predicted digit")
    ax[0].set_ylabel("true digit")
    ax[0].set_title(f"(a) 784-100-10 softmax network, test accuracy {sm['test_accuracy']:.4f}\n"
                    f"diagonal in blue; colour = off-diagonal count", fontsize=9)
    fig.colorbar(im, ax=ax[0], fraction=0.046)
    w = 0.38
    ax[1].bar(np.arange(10) - w / 2, prec, w, color=BLUE, label="precision (column)")
    ax[1].bar(np.arange(10) + w / 2, rec, w, color=RED, label="recall (row)")
    ax[1].set_ylim(0.9, 1.0)
    ax[1].set_xticks(range(10))
    ax[1].set_xlabel("digit")
    ax[1].legend(fontsize=8, loc="lower left")
    ax[1].set_title("(b) each class against the rest, from the same matrix", fontsize=9)
    fig.tight_layout()
    if name is not None:
        save_fig(name)
    return prec, rec

In [ ]:
#from src.plotting import fig_confusion10
# from week 43 Monday notebook

C = confusion_matrix_multiclass(y_test, pred_test_net, 10)
sm = {"C": C, "test_accuracy": accuracy_score(y_test, pred_test_net)} # Claude helped write this line
prec, rec = fig_confusion10(sm, name="e_confusion_matrix")

# the largest confusions
C_off = C.copy()
np.fill_diagonal(C_off, 0)
worst = np.dstack(np.unravel_index(np.argsort(-C_off.ravel())[:6], C_off.shape))[0]
print("largest confusions (true -> predicted: count): " + ", ".join(f"{i} -> {j}: {C_off[i, j]}" for i, j in worst))

# Comparing logistic regression to scikit and neural network to Pytorch

Finding the classifier accuracies using these libraries to compare to hand-backpropagation

In [ ]:
# Logistic regression vs scikit
# from Project 2 text

from sklearn.linear_model import LogisticRegression
# Initialize the model (multinomial logistic regression with the saga solver)
model = LogisticRegression(solver='saga', max_iter=1000, random_state=42)
# Train the model
model.fit(x_train, y_train)
from sklearn.metrics import accuracy_score
# Make predictions on the test set
y_pred = model.predict(x_test)
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

In [ ]:
# comparing network with hidden layer with PyTorch
# from week 44 Monday notebook, Claude helped debugg to correspond to our Network for comparison
import time

def torch_models(kind="mlp"):
    """The 784-100-10 network of week 41 ('mlp') as nn.Sequential."""
    import torch.nn as nn
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 100), nn.ReLU(), nn.Linear(100, 10))   # CHANGED: only the 'mlp' model


def torch_mnist(kind="mlp", epochs=5, batch=64, lr=1e-3, seed=1, verbose=True):
    """Train one of torch_models on the 60 000 MNIST images with Adam; test accuracy after every epoch.

    The data are standardised with the MNIST mean and standard deviation
    (0.1307, 0.3081), as the chapter's transforms.Normalize does; the loss is
    nn.CrossEntropyLoss, which applies the softmax itself, so the network ends
    with logits.
    """
    import torch
    import torch.nn as nn
    torch.manual_seed(seed)
    Xtr, ytr, Xte, yte = x_train, y_train.astype(int), x_test, y_test.astype(int)   # CHANGED: our arrays instead of mnist_arrays()
    Xtr = torch.tensor(((Xtr - 0.1307) / 0.3081).astype("float32")); Ytr = torch.tensor(ytr)
    Xte = torch.tensor(((Xte - 0.1307) / 0.3081).astype("float32")); Yte = torch.tensor(yte)
    net = torch_models(kind)
    n_par = sum(q.numel() for q in net.parameters())
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    hist, t0 = [], time.time()
    for ep in range(1, epochs + 1):
        net.train()                                  # dropout on
        perm = torch.randperm(len(Xtr))
        run_loss, nb = 0.0, 0
        for i in range(0, len(Xtr) - batch + 1, batch):
            idx = perm[i:i + batch]
            opt.zero_grad(set_to_none=True)
            loss = lossf(net(Xtr[idx]), Ytr[idx])
            loss.backward()                          # Eqs. (10.29)-(10.33), by autograd
            opt.step()
            run_loss += loss.item(); nb += 1
        net.eval()                                   # dropout off
        with torch.no_grad():
            pred = torch.cat([net(Xte[i:i + 1000]).argmax(1) for i in range(0, len(Xte), 1000)])
            acc = (pred == Yte).double().mean().item()
        hist.append((ep, run_loss / nb, acc, time.time() - t0))
        if verbose:
            print(f"  torch {kind:3s} epoch {ep}: loss {run_loss / nb:.4f}  test accuracy {acc:.4f}  ({time.time() - t0:.0f} s)")
    return {"history": np.array(hist), "n_params": n_par, "test_accuracy": hist[-1][2], "pred": pred.numpy()}   # CHANGED: removed the CNN-only outputs

In [ ]:
# from week 44 Monday notebook
r = torch_mnist("mlp")
print(f"  -> torch mlp: {r['n_params']} parameters, final test accuracy {r['test_accuracy']:.4f}")